In [1]:
import sys
import os
import time
import math

SRC_PATH = os.path.normpath(os.path.join(os.path.abspath(''), '..', 'src'))
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

from models.cpmp_transformer_v9 import CPMPTransformer
from training.training import load_model
from solvers.model_batched import BatchedModelSolver
from solvers.FRG import FRGSolver
from settings import INSTANCE_FOLDER

In [2]:
model = load_model(CPMPTransformer, 'v9_dataBSG_250k')
print(f"Modelo v9_dataBSG_250k cargado — H={model.hyperparams['H']}")

BATCH_SIZE = 40
BEAMS      = 5

solver_model = BatchedModelSolver(model, batch_size=BATCH_SIZE)
solver_bsg   = FRGSolver(beams=BEAMS)

print("Solvers listos:")
print(f"  - BSG250k (batched) : {solver_model.name} (batch_size={BATCH_SIZE})")
print(f"  - Solver BSG        : {solver_bsg.name} (beams={BEAMS})")

Modelo v9_dataBSG_250k cargado — H=12
Solvers listos:
  - BSG250k (batched) : BatchedModelSolver (batch_size=40)
  - Solver BSG        : FRG (beams=5)


In [3]:
def benchmark_bsg(solver, dat_files, H, max_steps):
    """Benchmark para FRGSolver — loop individual con timing por instancia."""
    solved_list, steps_list, time_list = [], [], []
    for filepath in dat_files:
        t0 = time.perf_counter()
        solved, steps = solver.solve_from_path(filepath, H, max_steps)
        time_list.append(time.perf_counter() - t0)
        solved_list.append(solved)
        steps_list.append(steps)
    return solved_list, steps_list, time_list


def avg(values, mask=None):
    if mask is not None:
        vals = [v for v, ok in zip(values, mask) if ok]
    else:
        vals = list(values)
    return sum(vals) / len(vals) if vals else math.nan


def pct_better_quality(avg_m, avg_b):
    if math.isnan(avg_m) or math.isnan(avg_b):
        return None, math.nan
    if avg_m < avg_b:
        return 'Modelo', (avg_b - avg_m) / avg_b * 100
    elif avg_b < avg_m:
        return 'SolverBSG', (avg_m - avg_b) / avg_m * 100
    return 'Empate', 0.0


def pct_faster(avg_tm, avg_tb):
    if math.isnan(avg_tm) or math.isnan(avg_tb):
        return None, math.nan
    if avg_tm < avg_tb:
        return 'Modelo', (avg_tb - avg_tm) / avg_tb * 100
    elif avg_tb < avg_tm:
        return 'SolverBSG', (avg_tm - avg_tb) / avg_tm * 100
    return 'Empate', 0.0

In [4]:
CVS_PATH  = INSTANCE_FOLDER / 'benchmarks' / 'CVS'
MAX_STEPS = 100

cvs_folders = sorted(
    [d for d in os.listdir(CVS_PATH) if (CVS_PATH / d).is_dir()]
)

results = {}

print(f"Corriendo benchmark en {len(cvs_folders)} categorías CVS...\n")

for folder_name in cvs_folders:
    H_real, S_real = [int(x) for x in folder_name.split('-')]
    H_inf       = H_real + 2
    folder_path = CVS_PATH / folder_name

    dat_files = sorted([
        str(folder_path / f)
        for f in os.listdir(folder_path)
        if f.endswith('.dat')
    ])

    print(f"  [{folder_name}] {len(dat_files)} instancias — H={H_inf}, S={S_real}, max_steps={MAX_STEPS}")

    # Modelo: solve_batch sobre todas las instancias de la carpeta a la vez
    solved_m, steps_m, time_m = solver_model.solve_batch(dat_files, H_inf, MAX_STEPS)

    # SolverBSG: loop individual
    solved_b, steps_b, time_b = benchmark_bsg(solver_bsg, dat_files, H_inf, MAX_STEPS)

    results[folder_name] = {
        'H': H_real, 'S': S_real, 'n': len(dat_files),
        'solved_m': solved_m, 'steps_m': steps_m, 'time_m': time_m,
        'solved_b': solved_b, 'steps_b': steps_b, 'time_b': time_b,
    }

print("\nBenchmark completado.")

Corriendo benchmark en 21 categorías CVS...

  [10-10] 40 instancias — H=12, S=10, max_steps=100


/Users/felipeastudillo/Desktop/Proyectos/Universidad/CPMP-Transformer/CPMP-Transformer/.venv/lib/python3.11/site-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


  [10-6] 40 instancias — H=12, S=6, max_steps=100
  [3-3] 40 instancias — H=5, S=3, max_steps=100
  [3-4] 40 instancias — H=5, S=4, max_steps=100
  [3-5] 40 instancias — H=5, S=5, max_steps=100
  [3-6] 40 instancias — H=5, S=6, max_steps=100
  [3-7] 40 instancias — H=5, S=7, max_steps=100
  [3-8] 40 instancias — H=5, S=8, max_steps=100
  [4-4] 40 instancias — H=6, S=4, max_steps=100
  [4-5] 40 instancias — H=6, S=5, max_steps=100
  [4-6] 40 instancias — H=6, S=6, max_steps=100
  [4-7] 40 instancias — H=6, S=7, max_steps=100
  [5-10] 40 instancias — H=7, S=10, max_steps=100
  [5-4] 40 instancias — H=7, S=4, max_steps=100
  [5-5] 40 instancias — H=7, S=5, max_steps=100
  [5-6] 40 instancias — H=7, S=6, max_steps=100
  [5-7] 40 instancias — H=7, S=7, max_steps=100
  [5-8] 40 instancias — H=7, S=8, max_steps=100
  [5-9] 40 instancias — H=7, S=9, max_steps=100
  [6-10] 40 instancias — H=8, S=10, max_steps=100
  [6-6] 40 instancias — H=8, S=6, max_steps=100

Benchmark completado.


In [5]:
# ─── Tabla por categoría ────────────────────────────────────────────────────
W       = 165
HDR_SEP = '═' * W

col_hdr = (
    f"{'Categoría':>10}  {'H':>3} {'S':>3} {'N':>4}  "
    f"{'── BSG250k (batched) ──':^38}  "
    f"{'── Solver BSG ──':^38}  "
    f"{'Calidad':^22}  {'Velocidad':^22}"
)
sub_hdr = (
    f"{'':>10}  {'':>3} {'':>3} {'':>4}  "
    f"{'resuelto':>10} {'avg pasos':>10} {'avg tiempo':>12}  "
    f"{'resuelto':>10} {'avg pasos':>10} {'avg tiempo':>12}  "
    f"{'ganador':^22}  {'ganador':^22}"
)

print(HDR_SEP)
print(col_hdr)
print(sub_hdr)
print(HDR_SEP)

for folder_name in cvs_folders:
    r = results[folder_name]
    n = r['n']

    nm = sum(r['solved_m'])
    nb = sum(r['solved_b'])

    both = [sm and sb for sm, sb in zip(r['solved_m'], r['solved_b'])]

    avg_steps_m      = avg(r['steps_m'], r['solved_m'])
    avg_steps_b      = avg(r['steps_b'], r['solved_b'])
    avg_steps_m_both = avg(r['steps_m'], both)
    avg_steps_b_both = avg(r['steps_b'], both)

    avg_time_m = avg(r['time_m'])
    avg_time_b = avg(r['time_b'])

    q_winner, q_pct = pct_better_quality(avg_steps_m_both, avg_steps_b_both)
    s_winner, s_pct = pct_faster(avg_time_m, avg_time_b)

    def fmt_steps(a):   return f"{a:>8.1f}" if not math.isnan(a) else "       -"
    def fmt_time(a):    return f"{a:>10.4f}s" if not math.isnan(a) else "          -"
    def fmt_winner(w, p):
        if w is None or math.isnan(p): return "       -"
        if w == 'Empate':              return "     Empate"
        return f"{w} +{p:5.1f}%"

    print(
        f"{folder_name:>10}  {r['H']:>3} {r['S']:>3} {n:>4}  "
        f"{nm:>3}/{n:<3} ({nm/n:>5.1%}) {fmt_steps(avg_steps_m)} {fmt_time(avg_time_m)}  "
        f"{nb:>3}/{n:<3} ({nb/n:>5.1%}) {fmt_steps(avg_steps_b)} {fmt_time(avg_time_b)}  "
        f"{fmt_winner(q_winner, q_pct):^22}  {fmt_winner(s_winner, s_pct):^22}"
    )

print(HDR_SEP)

═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
 Categoría    H   S    N         ── BSG250k (batched) ──                     ── Solver BSG ──                    Calidad                Velocidad       
                            resuelto  avg pasos   avg tiempo    resuelto  avg pasos   avg tiempo         ganador                 ganador        
═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════
     10-10   10  10   40    0/40  ( 0.0%)        -     0.0589s   40/40  (100.0%)    144.4     0.0576s                -           SolverBSG +  2.0%   
      10-6   10   6   40    0/40  ( 0.0%)        -     0.0420s   40/40  (100.0%)    106.3     0.0223s                -           SolverBSG + 46.9%   
       3-3    3   3   40   40/40  (100.0%)     10.1     0.0021s   40/4

In [6]:
# ─── Resumen global ─────────────────────────────────────────────────────────
all_solved_m = [s for r in results.values() for s in r['solved_m']]
all_steps_m  = [s for r in results.values() for s in r['steps_m']]
all_time_m   = [t for r in results.values() for t in r['time_m']]

all_solved_b = [s for r in results.values() for s in r['solved_b']]
all_steps_b  = [s for r in results.values() for s in r['steps_b']]
all_time_b   = [t for r in results.values() for t in r['time_b']]

total_n  = len(all_solved_m)
total_nm = sum(all_solved_m)
total_nb = sum(all_solved_b)

both_global = [sm and sb for sm, sb in zip(all_solved_m, all_solved_b)]
global_avg_steps_m      = avg(all_steps_m, all_solved_m)
global_avg_steps_b      = avg(all_steps_b, all_solved_b)
global_avg_steps_m_both = avg(all_steps_m, both_global)
global_avg_steps_b_both = avg(all_steps_b, both_global)

global_avg_time_m = avg(all_time_m)
global_avg_time_b = avg(all_time_b)

q_winner_g, q_pct_g = pct_better_quality(global_avg_steps_m_both, global_avg_steps_b_both)
s_winner_g, s_pct_g = pct_faster(global_avg_time_m, global_avg_time_b)

def quality_line(w, pct, am, ab):
    if w == 'Empate': return f"Empate — ambos promedian {am:.2f} pasos"
    return f"{w} es {pct:.1f}% mejor en calidad ({am:.2f} vs {ab:.2f} pasos)"

def speed_line(w, pct, tm, tb):
    if w == 'Empate': return f"Empate — ambos promedian {tm:.4f}s"
    return f"{w} es {pct:.1f}% más rápido ({tm:.4f}s vs {tb:.4f}s por instancia)"

BW = 60
print('═' * BW)
print(f"{'  RESUMEN GLOBAL':^{BW}}")
print('═' * BW)
print(f"  Instancias totales      : {total_n}")
print(f"  BSG250k (batched) res.  : {total_nm}/{total_n} ({total_nm/total_n:6.2%})")
print(f"  SolverBSG resueltas     : {total_nb}/{total_n} ({total_nb/total_n:6.2%})")
print()
print(f"  Avg pasos (propios resueltos)")
print(f"    BSG250k (batched) : {global_avg_steps_m:.2f} pasos")
print(f"    SolverBSG        : {global_avg_steps_b:.2f} pasos")
print(f"  Avg pasos (instancias resueltas por ambos)")
print(f"    BSG250k (batched) : {global_avg_steps_m_both:.2f} pasos")
print(f"    SolverBSG        : {global_avg_steps_b_both:.2f} pasos")
print(f"  Calidad   : {quality_line(q_winner_g, q_pct_g, global_avg_steps_m_both, global_avg_steps_b_both)}")
print()
print(f"  Avg tiempo por instancia")
print(f"    BSG250k (batched) : {global_avg_time_m:.4f}s")
print(f"    SolverBSG        : {global_avg_time_b:.4f}s")
print(f"  Velocidad : {speed_line(s_winner_g, s_pct_g, global_avg_time_m, global_avg_time_b)}")
print('═' * BW)

════════════════════════════════════════════════════════════
                        RESUMEN GLOBAL                      
════════════════════════════════════════════════════════════
  Instancias totales      : 840
  BSG250k (batched) res.  : 760/840 (90.48%)
  SolverBSG resueltas     : 840/840 (100.00%)

  Avg pasos (propios resueltos)
    BSG250k (batched) : 29.01 pasos
    SolverBSG        : 36.37 pasos
  Avg pasos (instancias resueltas por ambos)
    BSG250k (batched) : 29.01 pasos
    SolverBSG        : 27.01 pasos
  Calidad   : SolverBSG es 6.9% mejor en calidad (29.01 vs 27.01 pasos)

  Avg tiempo por instancia
    BSG250k (batched) : 0.0140s
    SolverBSG        : 0.0094s
  Velocidad : SolverBSG es 32.8% más rápido (0.0140s vs 0.0094s por instancia)
════════════════════════════════════════════════════════════
